In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Sobre o conjunto de dados - Comportamento na condução
*Contexto*

O principal fator de acidentes de trânsito é o comportamento agressivo na condução dos veículos. 
Conforme relatado pela Fundação AAA para Segurança no Trânsito durante um intervalo de tempo de quatro anos houveram 106.727 acidentes fatais que envolveram motoristas que cometeram uma ou mais ações agressivas na direção, isso equivale a 55,7% dos acidentes totais. 

Portanto, como prever o comportamento de condução perigosa com rapidez e precisão?

A condução agressiva inclui excesso de velocidade, frenagens repentinas e curvas repentinas à esquerda ou à direita. Todos esses eventos são refletidos nos dados do acelerômetro e do giroscópio. A partir desses dados dos sensores dremos realizar a análise das variáveis e em seguida aplicar modelos de ML para determinar suas classificações.

*Taxa de amostragem*: 2 amostras

Sensores: Acelerômetro e Giroscópio.

Dados:
* Aceleração (eixo X,Y,Z em metros por segundo ao quadrado (m/s2))
* Rotação (eixo X,Y, Z em graus por segundo (°/s))
* Classificação (SLOW, NORMAL, AGRESSIVE)
* Timestamp (tempo em segundos) - Tempo fornecido pelos sensores.


Comportamentos de condução: 
* Lento
* Normal
* Agressivo

**Importação das bibliotecas**

In [ ]:
import numpy as np 
import pandas as pd
import pandas
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
%matplotlib inline
import seaborn as sns; sns.set()

from sklearn import tree
import graphviz 
import os
import preprocessing 

import numpy as np 
import pandas as pd 
from plotly.offline import init_notebook_mode, iplot, plot
import plotly as py
init_notebook_mode(connected=True)
import plotly.graph_objs as go
from wordcloud import WordCloud
import matplotlib.pyplot as plt

from pandas_profiling import ProfileReport

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2, f_classif
from sklearn.model_selection import KFold
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import LinearSVC

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict

from sklearn.preprocessing import normalize
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.decomposition import PCA

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import CategoricalNB
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.model_selection import GridSearchCV

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier, XGBRFClassifier
from xgboost import plot_tree, plot_importance

from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, roc_curve
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis


import warnings
warnings.filterwarnings("ignore")

# **Lê e apresenta as informações sobre o Dataset

O Dataset é armazenado em um Dataframe Pandas**

In [ ]:
dataset = pandas.read_csv('../input/driving-behavior/test_motion_data.csv')
dataset.sample(10)


# Descrição das Variáveis

In [ ]:
dataset.info()

# Análise das Variáveis
* Variáveis Categóricas: ['Class']
* Variáveis Numéricas: ['AccX', 'AccY', 'AccZ', 'GyroX', 'GyroY', 'GyroZ', 'Timestamp']

# Variáveis Categóricas

In [ ]:
def bar_plot(variable):
    # get feature
    var = dataset[variable]
    # count number of categorical variable(value/sample)
    varValue = var.value_counts()
    
    # visualize
    plt.figure(figsize = (9,3))
    plt.bar(varValue.index, varValue)
    plt.xticks(varValue.index, varValue.index.values)
    plt.ylabel("Frequency")
    plt.title(variable)
    plt.show()
    print("{}:\n{}".format(variable,varValue))

In [ ]:
categorical = (dataset.dtypes == "object")
categorical_list = list(categorical[categorical].index)

print("Categorical variables:")
print(categorical_list)

In [ ]:
sns.set_style('darkgrid')
for c in categorical_list:
    bar_plot(c)

# Variáveis Numéricas

In [ ]:
numerical_float64 = (dataset.dtypes == "float64")
numerical_float64_list = list(numerical_float64[numerical_float64].index)

print("Numerical variables:")
print(numerical_float64_list)

numerical_int64 = (dataset.dtypes == "int64")
numerical_int64_list = list(numerical_int64[numerical_int64].index)

print("Numerical variables:")
print(numerical_int64_list)

# Distribuição em Histograma do dataset

In [ ]:
def plot_hist(variable):
    plt.figure(figsize = (9,3))
    plt.hist(dataset[variable], bins = 50)
    plt.xlabel(variable)
    plt.ylabel("Frequency")
    plt.title("{} Distribution with Histogram".format(variable))
    plt.show()

In [ ]:
for n in numerical_float64_list:
    plot_hist(n)
    
for n in numerical_int64_list:
    plot_hist(n)

# **Histograma**

In [ ]:
plt.figure(figsize=(20,20))


plt.subplot(4,2,1)
sns.histplot(dataset['AccX'], color = 'red', kde = True).set_title('Acceleration on X axis')

plt.subplot(4,2,2)
sns.histplot(dataset['AccY'], color = 'green', kde = True).set_title('Acceleration on Y axis')

plt.subplot(4,2,3)
sns.histplot(dataset['AccZ'], kde = True, color = 'blue').set_title('Acceleration on Z axis')

plt.subplot(4,2,4)
sns.histplot(dataset['GyroX'], kde = True, color = 'black').set_title('Rotation on X axis')

plt.subplot(4,2,5)
sns.histplot(dataset['GyroY'], kde = True, color = 'yellow').set_title('Rotation on Y axis')

plt.subplot(4,2,6)
sns.histplot(dataset['GyroZ'], kde = True, color = 'pink').set_title('Rotation on Z axis')

plt.subplot(4,2,7)
sns.histplot(dataset['Timestamp'], kde = True, color = 'purple').set_title('Unit of time provided by android sensors')


# **Correlação**

Quando falamos em análise de dados, muitas vezes precisamos entender qual é a associação entre duas ou mais variáveis. Nesse caso, a análise de correlação é uma forma descritiva que mede se há e qual o grau de dependência entre variáveis, ou seja, o quanto uma variável interfere em outra.

# **Tabela de Correlação do Dataset**

In [ ]:
dataset.corr()

# **Mapa de Correlação do Dataset**

In [ ]:
plt.figure(figsize=(12,8)) 
sns.heatmap(dataset.corr(), annot=True, cmap='Dark2_r', linewidths = 2)
plt.show()

In [ ]:
features = dataset.columns
sns.set_style('darkgrid')
sns.pairplot(dataset[features])

In [ ]:
sns.pairplot(dataset, hue = 'Class')

# **Visualização variáveis categóricas e numéricas**

**GRÁFICO BARPLOT**

In [ ]:
plt.figure(figsize=(20,15))
plt.subplot(4,2,1)
sns.barplot(x = 'Class', y = 'AccX', data = dataset, palette="cubehelix")
plt.subplot(4,2,2)
sns.barplot(x = 'Class', y = 'AccY', data = dataset, palette="Oranges")
plt.subplot(4,2,3)
sns.barplot(x = 'Class', y = 'AccZ', data = dataset, palette="Reds")
plt.subplot(4,2,4)
sns.barplot(x = 'Class', y = 'GyroX', data = dataset, palette="PuRd")
plt.subplot(4,2,5)
sns.barplot(x = 'Class', y = 'GyroY', data = dataset, palette="GnBu")
plt.subplot(4,2,6)
sns.barplot(x = 'Class', y = 'GyroZ', data = dataset, palette="Greens")
plt.subplot(4,2,7)
sns.barplot(x = 'Class', y = 'Timestamp', data = dataset, palette="Purples")



**GRÁFICO VIOLINPLOT**

In [ ]:
plt.figure(figsize=(20,15))
plt.subplot(4,2,1)
sns.violinplot(x = 'Class', y = 'AccX', data = dataset, palette="cubehelix")
plt.subplot(4,2,2)
sns.violinplot(x = 'Class', y = 'AccY', data = dataset, palette="Oranges")
plt.subplot(4,2,3)
sns.violinplot(x = 'Class', y = 'AccZ', data = dataset, palette="Reds")
plt.subplot(4,2,4)
sns.violinplot(x = 'Class', y = 'GyroX', data = dataset, palette="PuRd")
plt.subplot(4,2,5)
sns.violinplot(x = 'Class', y = 'GyroY', data = dataset, palette="GnBu")
plt.subplot(4,2,6)
sns.violinplot(x = 'Class', y = 'GyroZ', data = dataset, palette="Greens")
plt.subplot(4,2,7)
sns.violinplot(x = 'Class', y = 'Timestamp', data = dataset, palette="Purples")

**GRÁFICO BOXPLOT**

In [ ]:
plt.figure(figsize=(20,15))
plt.subplot(4,2,1)
sns.boxplot(x = 'Class', y = 'AccX', data = dataset, palette="cubehelix")
plt.subplot(4,2,2)
sns.boxplot(x = 'Class', y = 'AccY', data = dataset, palette="Oranges")
plt.subplot(4,2,3)
sns.boxplot(x = 'Class', y = 'AccZ', data = dataset, palette="Reds")
plt.subplot(4,2,4)
sns.boxplot(x = 'Class', y = 'GyroX', data = dataset, palette="PuRd")
plt.subplot(4,2,5)
sns.boxplot(x = 'Class', y = 'GyroY', data = dataset, palette="GnBu")
plt.subplot(4,2,6)
sns.boxplot(x = 'Class', y = 'GyroZ', data = dataset, palette="Greens")
plt.subplot(4,2,7)
sns.boxplot(x = 'Class', y = 'Timestamp', data = dataset, palette="Purples")


**GRÁFICO DISTPLOT**

**A curva mostra o gráfico de densidade que é essencialmente uma versão suave do histograma. O eixo y está em termos de densidade e o histograma é normalizado por padrão para que tenha a mesma escala y que o gráfico de densidade.**

In [ ]:
plt.figure(figsize=(20,15))
plt.subplot(4,2,1)
sns.distplot(dataset['AccX'], color="red").set_title('Acceleration on X axis')
plt.subplot(4,2,2)
sns.distplot(dataset['AccY'], color="green").set_title('Acceleration on Y axis')
plt.subplot(4,2,3)
sns.distplot(dataset['AccZ'], color="blue").set_title('Acceleration on Z axis')
plt.subplot(4,2,4)
sns.distplot(dataset['GyroX'], color="black").set_title('Rotation on X axis')
plt.subplot(4,2,5)
sns.distplot(dataset['GyroY'], color="orange").set_title('Rotation on Y axis.')
plt.subplot(4,2,6)
sns.distplot(dataset['GyroZ'], color="pink").set_title('Rotation on Z axis.')
plt.subplot(4,2,7)
sns.distplot(dataset['Timestamp'], color="purple").set_title('Unit of time provided by android sensors')

**GRÁFICO PIE**

In [ ]:
plt.figure(1, figsize=(5,5))
plt.title("Distribution of Class")
dataset['Class'].value_counts().plot.pie(autopct="%1.1f%%")

**Perfil Pandas**

Pandas profiling é uma biblioteca bem útil que gera relatórios sobre os dados. Com ele pode-se recuperar os tipos de dados, sua  distribuição e várias informações estatísticas. A ferramenta tem muitas técnicas para preapração dos dados. Bibliotecas gráficas envolvendo mapas de características e correlação. Mais detalhes em: https://pandas-profiling.github.io/pandas-profiling/docs/master/rtd/

In [ ]:
import pandas_profiling as pp
pp.ProfileReport(dataset)

# **Divisão entre Treinamento e Teste**

**X --> DADOS DE ENTRADA
Y --> DADOS DE SAÍDA**

In [ ]:
X = dataset.iloc[:,0:5].values 
y = dataset.iloc[:,6].values

**Criação de variáveis de entrada e saída para Teste e Treino**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=101) 
X_valid, X_test, y_valid, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

print(f'Total # of sample in whole dataset: {len(X)}')
print(f'Total # of sample in train dataset: {len(X_train)}')
print(f'Total # of sample in validation dataset: {len(X_valid)}')
print(f'Total # of sample in test dataset: {len(X_test)}')

**Standardization é um método que transforma os dados de forma a se ter média zero e desvio padrão de 1 e a distribuição tende a ser normal. A fórmula envolve a subtração do valor médio seguida pela divisão pela variança.**

In [ ]:
sc=StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.fit_transform(X_test)

# **Score dos Modelos**

**Aplicam-se algoritmos de ML ao dataset. Os resultados conterão scores de treinamento, teste e validação, matriz de confusão, informações estatísticas e relatórios de classificação para cada algoritmo.**

In [ ]:
models = {
    'GaussianNB': GaussianNB(),
    'BernoulliNB': BernoulliNB(),
    'LogisticRegression': LogisticRegression(),
    'RandomForestClassifier': RandomForestClassifier(),
    'SupportVectorMachine': SVC(),
    'DecisionTreeClassifier': DecisionTreeClassifier(),
    'KNeighborsClassifier': KNeighborsClassifier(),
    'GradientBoostingClassifier': GradientBoostingClassifier(),
    'Stochastic Gradient Descent':  SGDClassifier(max_iter=5000, random_state=0),
    'Neural Nets': MLPClassifier(solver='lbfgs', alpha=1e-5, hidden_layer_sizes=(5000, 10), random_state=1),
}

modelNames = ["GaussianNB", 'BernoulliNB','LogisticRegression','RandomForestClassifier','SupportVectorMachine',
             'DecisionTreeClassifier', 'KNeighborsClassifier','GradientBoostingClassifier',
             'Stochastic Gradient Descent', 'Neural Nets']

trainScores = []
validationScores = []
testScores = []

for m in models:
  model = models[m]
  model.fit(X_train, y_train)
  score = model.score(X_valid, y_valid)
  #print(f'{m} validation score => {score*100}')
    
  print(f'{m}') 
  train_score = model.score(X_train, y_train)
  print(f'Train score of trained model: {train_score*100}')
  trainScores.append(train_score*100)

  validation_score = model.score(X_valid, y_valid)
  print(f'Validation score of trained model: {validation_score*100}')
  validationScores.append(validation_score*100)

  test_score = model.score(X_test, y_test)
  print(f'Test score of trained model: {test_score*100}')
  testScores.append(test_score*100)
  print(" ")
    
  y_predictions = model.predict(X_test)
  conf_matrix = confusion_matrix(y_predictions, y_test)

  print(f'Confussion Matrix: \n{conf_matrix}\n')

  predictions = model.predict(X_test)
  cm = confusion_matrix(predictions, y_test)

  tn = conf_matrix[0,0]
  fp = conf_matrix[0,1]
  tp = conf_matrix[1,1]
  fn = conf_matrix[1,0]
  accuracy  = (tp + tn) / (tp + fp + tn + fn)
  precision = tp / (tp + fp)
  recall    = tp / (tp + fn)
  f1score  = 2 * precision * recall / (precision + recall)
  specificity = tn / (tn + fp)
  print(f'Accuracy : {accuracy}')
  print(f'Precision: {precision}')
  print(f'Recall   : {recall}')
  print(f'F1 score : {f1score}')
  print(f'Specificity : {specificity}')
  print("") 
  print(f'Classification Report: \n{classification_report(predictions, y_test)}\n')
  print("")
   
  for m in range (1):
    current = modelNames[m]
    modelNames.remove(modelNames[m])

  preds = model.predict(X_test)
  confusion_matr = confusion_matrix(y_test, preds) #normalize = 'true'
  print("############################################################################")
  print("")
  print("")
  print("")

# **PONTUAÇÃO DE TESTE DOS MODELOS**

In [ ]:
plt.figure(figsize=(20,10))
sns.set_style('darkgrid')
plt.title('Train - Validation - Test Scores of Models', fontweight='bold', size = 24)

barWidth = 0.25
 
bars1 = trainScores
bars2 = validationScores
bars3 = testScores
 
r1 = np.arange(len(bars1))
r2 = [x + barWidth for x in r1]
r3 = [x + barWidth for x in r2]
 
plt.bar(r1, bars1, color='blue', width=barWidth, edgecolor='white', label='train', yerr=0.5,ecolor="black",capsize=10)
plt.bar(r2, bars2, color='#557f2d', width=barWidth, edgecolor='white', label='validation', yerr=0.5,ecolor="black",capsize=10, alpha = .50)
plt.bar(r3, bars3, color='red', width=barWidth, edgecolor='white', label='test', yerr=0.5,ecolor="black",capsize=10, hatch = '-')
 
modelNames = ["GaussianNB", 'BernoulliNB','LogisticRegression','RandomForestClassifier','SupportVectorMachine',
             'DecisionTreeClassifier', 'KNeighborsClassifier','GradientBoostingClassifier',
             'Stochastic Gradient Descent', 'Neural Nets']
    
plt.xlabel('Algorithms', fontweight='bold', size = 24)
plt.ylabel('Scores', fontweight='bold', size = 24)
plt.xticks([r + barWidth for r in range(len(bars1))], modelNames, rotation = 75)
 
plt.legend()
plt.show()

# **ACURÁCIA DOS MODELOS TREINADOS**

In [ ]:
for i in range(10):
    print(f'Accuracy of {modelNames[i]} -----> {testScores[i]}')